# 🚀 SPARK TUTORIAL 01: INTRODUCCIÓN BÁSICA

## 🎯 **OBJETIVO**
Aprender los conceptos fundamentales de Apache Spark

## 📋 **CONTENIDO**
- ¿Qué es Apache Spark?
- Configuración de SparkSession
- Conceptos básicos: RDD, DataFrames, Datasets
- Primeras operaciones
- Lazy Evaluation
- Trabajo con archivos

---

## 🔧 **CONFIGURACIÓN INICIAL**


In [ ]:
# Importar librerías necesarias
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum as spark_sum
import os

print("📚 Librerías importadas correctamente")


## 🔧 **PASO 1: CREAR SPARKSESSION**

SparkSession es el punto de entrada principal para usar Spark SQL y DataFrames.


In [ ]:
# Crear SparkSession para el curso
import socket
import os

# Detectar si estamos dentro de un contenedor Docker
def get_spark_master():
    try:
        # Intentar conectar al master de Docker
        hostname = socket.gethostname()
        if 'jupyter' in hostname or 'master' in hostname:
            return "spark://master:7077"  # Desde dentro del contenedor
        else:
            return "spark://localhost:7077"  # Desde fuera del contenedor
    except:
        return "local[*]"  # Fallback a modo local

spark_master_url = get_spark_master()
print(f"🔧 Conectando a: {spark_master_url}")

spark = SparkSession.builder \
    .appName("EducacionIT-Spark-Basics") \
    .master(spark_master_url) \
    .config("spark.executor.memory", "800m") \
    .config("spark.executor.cores", "1") \
    .config("spark.executor.instances", "2") \
    .config("spark.driver.memory", "1g") \
    .config("spark.driver.cores", "1") \
    .config("spark.sql.adaptive.enabled", "false") \
    .config("spark.sql.warehouse.dir", "/user/hive/warehouse") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .enableHiveSupport() \
    .getOrCreate()

print("✅ SparkSession creada exitosamente")


## 📊 **PASO 2: INFORMACIÓN DEL CLUSTER**

Vamos a ver información sobre nuestro cluster Spark.


In [ ]:
# Información de la aplicación
print("🔧 Nombre de la aplicación:", spark.sparkContext.appName)
print("🆔 ID de la aplicación:", spark.sparkContext.applicationId)
print("🌐 URL de la UI:", spark.sparkContext.uiWebUrl)
print("📦 Versión de Spark:", spark.version)
print("🐍 Versión de Python:", spark.sparkContext.pythonVer)


In [ ]:
# Información del cluster
status = spark.sparkContext.statusTracker()
executors = status.getExecutorInfos()

print(f"👥 Número de ejecutores: {len(executors)}")
print("\n💾 Información de ejecutores:")
for executor in executors:
    print(f"  - Ejecutor {executor.executorId}: {executor.maxMemory} bytes")


## 📚 **PASO 3: TRABAJANDO CON RDDS**

Los RDDs (Resilient Distributed Datasets) son la abstracción fundamental de Spark.


In [ ]:
# Crear RDD desde una lista
numeros = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
rdd_numeros = spark.sparkContext.parallelize(numeros)

print(f"🔢 RDD creado con {rdd_numeros.count()} elementos")
print(f"➕ Suma total: {rdd_numeros.sum()}")
print(f"📊 Promedio: {rdd_numeros.mean():.2f}")
print(f"📈 Máximo: {rdd_numeros.max()}")
print(f"📉 Mínimo: {rdd_numeros.min()}")


In [ ]:
# Transformaciones con RDD
rdd_pares = rdd_numeros.filter(lambda x: x % 2 == 0)
print(f"🔢 Números pares: {rdd_pares.collect()}")

rdd_cuadrados = rdd_numeros.map(lambda x: x ** 2)
print(f"📐 Cuadrados: {rdd_cuadrados.collect()}")


## 📊 **PASO 4: TRABAJANDO CON DATAFRAMES**

Los DataFrames son más fáciles de usar que los RDDs y están optimizados.


In [ ]:
# Crear DataFrame desde datos de ejemplo
datos_empleados = [
    ("Juan", "IT", 50000, 25),
    ("María", "Marketing", 45000, 30),
    ("Carlos", "IT", 55000, 28),
    ("Ana", "HR", 40000, 35),
    ("Luis", "IT", 60000, 32),
    ("Laura", "Marketing", 48000, 27)
]

columnas = ["nombre", "departamento", "salario", "edad"]
df_empleados = spark.createDataFrame(datos_empleados, columnas)

print("👥 DataFrame de empleados creado:")
df_empleados.show()


In [ ]:
# Ver schema del DataFrame
print("📋 Schema del DataFrame:")
df_empleados.printSchema()


In [ ]:
# Operaciones básicas con DataFrames
print("🔍 EMPLEADOS DEL DEPARTAMENTO IT:")
df_empleados.filter(col("departamento") == "IT").show()

print("\n💰 EMPLEADOS CON SALARIO > 50000:")
df_empleados.filter(col("salario") > 50000).show()


In [ ]:
# Agregaciones
print("📊 ESTADÍSTICAS POR DEPARTAMENTO:")
df_empleados.groupBy("departamento").agg(
    count("nombre").alias("total_empleados"),
    spark_sum("salario").alias("salario_total"),
    spark_sum("edad").alias("edad_total")
).show()


## ⚡ **PASO 5: LAZY EVALUATION**

Spark usa evaluación perezosa (lazy evaluation) para optimizar el procesamiento.


In [ ]:
# Crear DataFrame grande para demostrar lazy evaluation
numeros = list(range(1, 1001))
df_numeros = spark.createDataFrame([(x,) for x in numeros], ["numero"])

print("🔢 DataFrame con 1000 números creado")

# Transformaciones (lazy)
df_transformado = df_numeros \
    .filter(col("numero") % 2 == 0) \
    .filter(col("numero") > 100) \
    .filter(col("numero") < 500) \
    .select((col("numero") * 2).alias("doble"))

print("🔄 Transformaciones aplicadas (sin ejecutar)")
print("⚡ Las transformaciones son 'lazy' - no se ejecutan hasta ahora")


In [ ]:
# Acción (triggers execution)
print("🚀 Ejecutando acción - esto dispara el procesamiento...")
import time
start_time = time.time()

resultado = df_transformado.collect()

end_time = time.time()
print(f"✅ Resultado: {len(resultado)} elementos")
print(f"📊 Primeros 10: {[r.doble for r in resultado[:10]]}")
print(f"⏱️ Tiempo de ejecución: {end_time - start_time:.3f} segundos")


## 🎯 **RESUMEN DEL TUTORIAL**

¡Felicitaciones! Has completado el tutorial básico de Spark.

### **📚 Conceptos aprendidos:**
- ✅ **SparkSession**: Punto de entrada principal
- ✅ **RDDs**: Estructura de datos distribuida básica
- ✅ **DataFrames**: Estructura optimizada para análisis
- ✅ **Transformaciones**: Operaciones que crean nuevos DataFrames
- ✅ **Acciones**: Operaciones que ejecutan el procesamiento
- ✅ **Lazy Evaluation**: Optimización automática de Spark

### **🚀 Próximos pasos:**
1. **Tutorial 02**: DataFrames Avanzados
2. **Tutorial 03**: Spark SQL
3. **Experimentar** con tus propios datos

---

**🎉 ¡Has dominado los fundamentos de Apache Spark!**


In [ ]:
# Cerrar SparkSession
spark.stop()
print("🔒 SparkSession cerrada correctamente")
